# Kaipuleohone archive

Scrapes resource metadata and sizes from the Kaipuleohone language archive.

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [ ]:
hawaii = 'https://scholarspace.manoa.hawaii.edu/communities/533eb99e-2e59-47a5-9c4e-652352bfc5a6'

In [ ]:
hawaii = requests.get(hawaii).text

In [ ]:
hawaii_soup = BeautifulSoup(hawaii, 'html.parser')

In [ ]:
def extract_hrefs(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    hrefs = [a['href'] for a in soup.find_all('a', class_='lead ng-star-inserted', href=True)]
    return hrefs

In [ ]:
base_url = 'https://scholarspace.manoa.hawaii.edu/communities/533eb99e-2e59-47a5-9c4e-652352bfc5a6?cmcl.page='

# List to store all hrefs
all_hrefs = []

# Iterate over pages
for page_number in range(1, 10):  # Adjust the range as needed
    page_url = f'{base_url}{page_number}'
    hrefs = extract_hrefs(page_url)
    all_hrefs.extend(hrefs)

# Print all hrefs
for href in all_hrefs:
    print(href)

In [ ]:
prefix = 'https://scholarspace.manoa.hawaii.edu'
for i in range(len(all_hrefs)):
    all_hrefs[i] = prefix + all_hrefs[i]

In [ ]:
def extract_hrefs(url, max_pages=30):
    all_hrefs = []
    visited_hrefs = set()
    page_num = 1
    while page_num <= max_pages:
        print(page_num)
        response = requests.get(url + f'?cp.page={page_num}')
        # Check if the response status code indicates success (200)
        if response.status_code != 200:
            break
        soup = BeautifulSoup(response.text, 'html.parser')
        links = soup.find_all('a', {'class': 'lead item-list-title dont-break-out ng-star-inserted'})
        inner_hrefs = [link['href'] for link in links]
        if not inner_hrefs:
            break
        for href in inner_hrefs:
            if href in visited_hrefs:
                # If we encounter a href that we've already visited, break the loop
                break
            all_hrefs.append(href)
            visited_hrefs.add(href)
        else:
            page_num += 1
            continue
        break  # Break the while loop if we encounter a visited href
    return all_hrefs


# Iterate over each webpage URL and extract href attributes
inner_hrefs = []
for url in chunk_5:
    hrefs = extract_hrefs(url)
    inner_hrefs.extend(hrefs)

In [ ]:
for i in range(len(inner_hrefs)):
    inner_hrefs[i] = prefix + inner_hrefs[i] + '/full'

In [ ]:
# Function to scrape content language and code from inner_href
def scrape_content_language(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    language_tag = soup.find('td', text='dc.content.language')
    language_code_tag = soup.find('td', text='dc.content.languagecode')
    if language_tag and language_code_tag:
        language = language_tag.find_next_sibling('td').text.strip()
        language_code = language_code_tag.find_next_sibling('td').text.strip()
        return language, language_code
    else:
        return None, None

# Function to scrape file name and size
def scrape_files_info(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    file_sections = soup.find_all('div', class_='file-section row mb-3 ng-star-inserted')
    file_names = []
    file_sizes = []
    for section in file_sections:
        name_tag = section.find('dt', text='Name:')
        size_tag = section.find('dt', text='Size:')
        if name_tag and size_tag:
            file_names.append(name_tag.find_next_sibling('dd').text.strip())
            file_sizes.append(size_tag.find_next_sibling('dd').text.strip())
    return file_names, file_sizes

In [ ]:
# Initialize lists to store scraped data
languages = []
language_codes = []
file_names_list = []
file_sizes_list = []

# Scrape data for each inner href
for index, href in enumerate(inner_hrefs):
    print(index)
    language, language_code = scrape_content_language(href)
    if language and language_code:
        languages.append(language)
        language_codes.append(language_code)
        file_names, file_sizes = scrape_files_info(href)
        file_names_list.append(file_names)
        file_sizes_list.append(file_sizes)
        origin_href = href

# Create DataFrame
data = {
    'Href': origin_href,
    'Content Language': languages,
    'Language Code': language_codes,
    'File Name': file_names_list ,
    'File Size': file_sizes_list 
}

results = pd.DataFrame(data)
results = results[results['File Size'] != '[]']

In [ ]:
unit_multipliers = {
        'B': 1,
        'KB': 1024,
        'MB': 1024 ** 2,
        'GB': 1024 ** 3
    }

def convert_to_bytes(size_str):
    # Remove brackets and split into number and unit
    size_str = size_str.strip('[]')
    size_value, size_unit = size_str.split()
    size_value = float(size_value)
    
    # Convert to bytes
    bytes_size = int(size_value * unit_multipliers[size_unit])
    return bytes_size

def convert_list_to_total_bytes(size_list):
    total_bytes = sum(convert_to_bytes(size_str) for size_str in size_list)
    return total_bytes

bytes_to_gigabytes = lambda x: x / (1024 ** 3)

# Apply the conversion to the DataFrame column
results['Size_in_Bytes'] = results['File Size'].apply(convert_list_to_total_bytes)

In [ ]:
def convert_to_bytes(size_str):
    # Conversion factors
    unit_multipliers = {
        'B': 1,
        'KB': 1024,
        'MB': 1024 ** 2,
        'GB': 1024 ** 3
    }
    
    # Remove brackets if present and split into number and unit
    size_str = size_str.strip('[]')
    size_parts = size_str.split()
    
    # Check if there are enough parts to unpack
    if len(size_parts) != 2:
        return None
    
    size_value, size_unit = size_parts
    
    try:
        size_value = float(size_value)
    except ValueError:
        return None
    
    # Convert to bytes
    bytes_size = int(size_value * unit_multipliers[size_unit])
    return bytes_size


def convert_list_to_total_bytes(size_list):
    total_bytes = sum(convert_to_bytes(size_str) for size_str in size_list)
    return total_bytes

In [ ]:
results['Size_in_Bytes'] = results['File Size'].apply(convert_list_to_total_bytes)

In [ ]:
results['gigabytes_column'] = results['Size_in_Bytes'].apply(bytes_to_gigabytes)

In [ ]:
results =  results.groupby('Content Language')['gigabytes_column'].sum().reset_index()

In [ ]:
results = results.sort_values(by='gigabytes_column', ascending=False)

In [ ]:
results

In [ ]:
results =

In [ ]:
len(data['Content Language'])

In [ ]:
data['Content Language']

In [ ]:
len(data['File Size'])